# 05 — Model Monitoring & Observability

Dashboard de monitoramento do modelo de detecção de fraude com boas práticas de MLOps.

**Métricas de Observabilidade (corrigidas em Out/2026):**
- **Latência por requisição HTTP**: avg, p50, p95, p99 (ms) — `AVG(latency_ms)`, `PERCENTILE(latency_ms, ...)` sobre requisições individuais
- **Latência por predição**: `avg_latency_per_record_ms = SUM(latency_ms) / SUM(n_records)` — normalizada por predição
- **Cold vs Warm Start**: dimensão `start_type` via `LAG()` — Cold = primeira requisição após 5+ min de inatividade
- **Total de predições**: `SUM(n_records)` — 2.000.000 predições (corrigido de `COUNT(*)` que contava 2.000 requisições)
- **Total de requisições HTTP**: `COUNT(*)` — 2.000 (renomeado de `total_batches = COUNT(DISTINCT event_timestamp)`)
- **Error rate**: % de requisições HTTP com `status != 'success'`

**Métricas de Drift:**
- **Data drift**: KS test (features numéricas) + PSI (features categóricas)
- **Prediction drift**: distribuição de `probability_fraud` treino vs produção

**Resultados atuais:**
| Métrica | Valor |
|---|---|
| Latência média/requisição | 19.034 ms |
| Latência média/predição | 19.03 ms |
| Cold Start | 36.964 ms |
| Warm Start | 19.025 ms |
| Error rate | 0% |

**Limitação:** A tabela não separa Feature Lookup / Model Execution — o modelo v13 usa features diretas (sem FeatureLookup), e `scoring.py` rastreia apenas `latency_ms` único.

**Fontes:**
- Treino: `gold.fraud_features` (split='train')
- Produção: `gold.model_predictions`
- Latência: `bronze.monitoring_raw_events` (8 colunas: event_timestamp, TransactionID, probability_fraud, is_fraud, latency_ms, endpoint_name, n_records, status)

In [0]:
import sys
import pandas as pd
import numpy as np
from scipy.stats import ks_2samp

sys.path.insert(0, "/Workspace/Users/eduardolimaofficial410@gmail.com/databricks-fraud-detection/src")
from databricks_fraud_model.features import MODEL_FEATURES
from databricks_fraud_model.utils import load_config

cfg = load_config()
CATALOG = cfg["catalog"]

CATEGORICAL_FEATURES = [
    'ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain',
    'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9',
    'id_12', 'id_16', 'id_31', 'id_36', 'id_37', 'id_38',
    'DeviceType', 'DeviceInfo',
    'DeviceType_isnull', 'TransactionAmt_round',
    'is_morning', 'is_afternoon', 'is_night', 'is_weekend',
    'email_match', 'P_email_free', 'R_email_free', 'device_mismatch',
]
NUMERIC_FEATURES = [f for f in MODEL_FEATURES if f not in CATEGORICAL_FEATURES]

In [0]:
df_train = spark.sql(f"""SELECT * FROM {CATALOG}.gold.fraud_features WHERE split = 'train'""").toPandas()
df_prod = spark.sql(f"""SELECT * FROM {CATALOG}.gold.model_predictions""").toPandas()

try:
    df_latency = spark.sql(f"""
        SELECT * FROM {CATALOG}.bronze.monitoring_raw_events
        ORDER BY event_timestamp DESC
    """).toPandas()
    has_latency = True
except Exception:
    has_latency = False

In [0]:
# Data Drift — features numericas (KS test)
# KS statistic: 0 = distribuicoes identicas, 1 = completamente diferentes
# p-valor < 0.05 indica drift significativo

ks_results = []
for col in NUMERIC_FEATURES:
    if col not in df_train.columns or col not in df_prod.columns:
        continue
    train_vals = df_train[col].dropna().values
    prod_vals = df_prod[col].dropna().values
    
    if len(train_vals) < 10 or len(prod_vals) < 10:
        continue
    
    statistic, p_value = ks_2samp(train_vals, prod_vals)
    ks_results.append({
        'feature': col,
        'ks_statistic': round(statistic, 4),
        'p_value': p_value,
        'drift': p_value < 0.05,
    })

ks_df = pd.DataFrame(ks_results).sort_values('ks_statistic', ascending=False)
drifted_numeric = ks_df[ks_df['drift']]
display(ks_df.head(15))

In [0]:
# ============================================================================
# OBSERVABILIDADE — Latência por requisição HTTP, throughput e error rate
# Corrigido em Out/2026: metricas por requisicao HTTP individual
# ============================================================================
if has_latency:
    print("=" * 60)
    print("           ENDPOINT LATENCY & OBSERVABILITY")
    print("=" * 60)
    
    lat = df_latency['latency_ms']
    total_requests = len(df_latency)  # COUNT(*) — total de requisicoes HTTP
    total_predictions = df_latency['n_records'].sum()  # SUM(n_records) — predicoes reais
    avg_latency_per_record = df_latency['latency_ms'].sum() / total_predictions  # latencia por predicao
    
    print(f"\nTotal de requisicoes HTTP: {total_requests:,}")
    print(f"Total de predicoes (SUM n_records): {total_predictions:,}")
    print(f"Endpoint: {df_latency['endpoint_name'].iloc[0]}")
    print(f"\n--- Latência por Requisição HTTP (ms) ---")
    print(f"  Avg:   {lat.mean():.2f} ms")
    print(f"  P50:   {lat.median():.2f} ms")
    print(f"  P95:   {lat.quantile(0.95):.2f} ms")
    print(f"  P99:   {lat.quantile(0.99):.2f} ms")
    print(f"  Min:   {lat.min():.2f} ms")
    print(f"  Max:   {lat.max():.2f} ms")
    print(f"\n--- Latência por Predição (ms) ---")
    print(f"  Avg:   {avg_latency_per_record:.4f} ms/predicao")
    
    # Cold vs Warm Start (LAG sobre event_timestamp por endpoint)
    df_latency_sorted = df_latency.sort_values(['endpoint_name', 'event_timestamp']).copy()
    df_latency_sorted['prev_ts'] = df_latency_sorted.groupby('endpoint_name')['event_timestamp'].shift(1)
    df_latency_sorted['gap_min'] = (df_latency_sorted['event_timestamp'] - df_latency_sorted['prev_ts']).dt.total_seconds() / 60
    df_latency_sorted['start_type'] = df_latency_sorted['gap_min'].apply(
        lambda x: 'Cold Start' if pd.isna(x) or x > 5 else 'Warm Start'
    )
    
    print(f"\n--- Cold vs Warm Start ---")
    for st in ['Cold Start', 'Warm Start']:
        subset = df_latency_sorted[df_latency_sorted['start_type'] == st]
        if len(subset) > 0:
            print(f"  {st}: {len(subset)} requisicoes | avg latency: {subset['latency_ms'].mean():.2f} ms")
    
    # Throughput
    print(f"\n--- Throughput ---")
    print(f"  Requisicoes HTTP: {total_requests:,}")
    print(f"  Predicoes totais: {total_predictions:,}")
    print(f"  Predicoes por requisicao: {total_predictions / total_requests:.1f}")
    
    # Error rate
    error_rate = (df_latency['status'] != 'success').mean()
    print(f"\n--- Error Rate ---")
    print(f"  Erros: {(df_latency['status'] != 'success').sum()}/{total_requests} ({error_rate:.2%})")
    
    # Alertas de latencia
    print(f"\n--- Alertas ---")
    if lat.quantile(0.95) > 500:
        print(f"ALERTA: P95 > 500ms — endpoint lento")
    elif lat.quantile(0.95) > 200:
        print(f"ATENCAO: P95 > 200ms — monitorar performance")
    else:
        print(f"OK: P95 < 200ms — latencia aceitavel")
    
    display(df_latency[['TransactionID', 'probability_fraud', 'latency_ms', 'n_records', 'event_timestamp']].head(10))
else:
    print("Dados de latencia indisponiveis — execute 04_scoring para popular bronze.monitoring_raw_events")

In [0]:
# Data Drift — features categoricas (PSI - Population Stability Index)
# PSI < 0.1: sem drift | 0.1-0.25: drift moderado | > 0.25: drift significativo

def calculate_psi(train_dist, prod_dist):
    """Calcula PSI para distribuicoes categoricas."""
    all_categories = set(train_dist.index) | set(prod_dist.index)
    train_pct = np.array([train_dist.get(cat, 0) for cat in all_categories], dtype=float)
    prod_pct = np.array([prod_dist.get(cat, 0) for cat in all_categories], dtype=float)
    
    train_pct = train_pct / train_pct.sum()
    prod_pct = prod_pct / prod_pct.sum()
    
    epsilon = 1e-6
    train_pct = np.where(train_pct == 0, epsilon, train_pct)
    prod_pct = np.where(prod_pct == 0, epsilon, prod_pct)
    
    return np.sum((prod_pct - train_pct) * np.log(prod_pct / train_pct))

psi_results = []
for col in CATEGORICAL_FEATURES:
    if col not in df_train.columns or col not in df_prod.columns:
        continue
    
    train_dist = df_train[col].value_counts(normalize=True)
    prod_dist = df_prod[col].value_counts(normalize=True)
    
    psi = calculate_psi(train_dist, prod_dist)
    psi_results.append({
        'feature': col,
        'psi': round(psi, 4),
        'status': 'significant' if psi > 0.25 else ('moderate' if psi > 0.1 else 'stable'),
    })

psi_df = pd.DataFrame(psi_results).sort_values('psi', ascending=False)
drifted_categorical = psi_df[psi_df['psi'] > 0.25]
display(psi_df.head(15))

In [0]:
# Prediction Drift — distribuicao de probability_fraud
print("=== Prediction Drift ===")
print(f"Production predictions: {len(df_prod):,}")
print(f"Mean probability_fraud: {df_prod['probability_fraud'].mean():.4f}")
print(f"Median probability_fraud: {df_prod['probability_fraud'].median():.4f}")
print(f"Std: {df_prod['probability_fraud'].std():.4f}")
print(f"Predicted fraud rate (is_fraud=1): {df_prod['is_fraud'].mean():.4f}")
print(f"Score range: [{df_prod['probability_fraud'].min():.4f}, {df_prod['probability_fraud'].max():.4f}]")

# Se houver target no treino, comparar taxa de fraude
if 'target' in df_train.columns:
    train_fraud_rate = df_train['target'].mean()
    prod_fraud_rate = df_prod['is_fraud'].mean()
    print(f"\nTrain fraud rate: {train_fraud_rate:.4f}")
    print(f"Prod predicted fraud rate: {prod_fraud_rate:.4f}")
    print(f"Difference: {abs(prod_fraud_rate - train_fraud_rate):.4f}")

display(df_prod['probability_fraud'].describe())

In [0]:
# ============================================================================
# SUMMARY — Dashboard consolidado de monitoring
# ============================================================================
print("=" * 60)
print("         MODEL MONITORING DASHBOARD")
print("=" * 60)

# --- 1. Observabilidade ---
print("\n1. OBSERVABILIDADE")
if has_latency:
    lat = df_latency['latency_ms']
    total_requests = len(df_latency)
    total_predictions = df_latency['n_records'].sum()
    avg_per_record = df_latency['latency_ms'].sum() / total_predictions
    print(f"   Latencia/requisicao: avg={lat.mean():.0f}ms | P50={lat.median():.0f}ms | P95={lat.quantile(0.95):.0f}ms | P99={lat.quantile(0.99):.0f}ms")
    print(f"   Latencia/predicao: {avg_per_record:.2f} ms")
    print(f"   Requisicoes HTTP: {total_requests:,} | Predicoes: {total_predictions:,}")
    print(f"   Error rate: {(df_latency['status'] != 'success').mean():.2%}")
else:
    print("   Dados de latencia indisponiveis")

# --- 2. Data Drift ---
print("\n2. DATA DRIFT")
# Combina resultados
all_drift = []
for _, row in ks_df.iterrows():
    all_drift.append({'feature': row['feature'], 'metric': 'KS', 'value': row['ks_statistic'], 'p_value': row['p_value'], 'drifted': row['drift']})
for _, row in psi_df.iterrows():
    all_drift.append({'feature': row['feature'], 'metric': 'PSI', 'value': row['psi'], 'p_value': None, 'drifted': row['psi'] > 0.25})

drift_summary = pd.DataFrame(all_drift).sort_values(['drifted', 'value'], ascending=[False, False])
total_drifted = drift_summary['drifted'].sum()
total_tested = len(drift_summary)
drift_rate = total_drifted / total_tested if total_tested > 0 else 0
print(f"   Features testadas: {total_tested} | Com drift: {total_drifted} ({drift_rate:.1%})")
print(f"   - Numericas (KS p<0.05): {len(drifted_numeric)}")
print(f"   - Categoricas (PSI>0.25): {len(drifted_categorical)}")

# --- 3. Prediction Drift ---
print("\n3. PREDICTION DRIFT")
print(f"   Prod: {len(df_prod):,} predicoes | Mean score: {df_prod['probability_fraud'].mean():.4f}")
if 'target' in df_train.columns:
    print(f"   Train fraud rate: {df_train['target'].mean():.4f} | Prod predicted: {df_prod['is_fraud'].mean():.4f}")

# --- 4. Alertas ---
print("\n4. ALERTAS")
alerts = []
if has_latency and df_latency['latency_ms'].quantile(0.95) > 500:
    alerts.append("P95 > 500ms — endpoint lento")
if drift_rate > 0.20:
    alerts.append(f"{drift_rate:.1%} das features com drift — RETREINAR")
elif drift_rate > 0.10:
    alerts.append(f"{drift_rate:.1%} das features com drift — monitorar")
if not alerts:
    print("   Tudo OK — modelo estavel")
else:
    for a in alerts:
        print(f"   ALERTA: {a}")

# Top drifted features
if total_drifted > 0:
    print("\n--- Top 10 Drifted Features ---")
    display(drift_summary[drift_summary['drifted']].head(10))